# TP2 — Tries e Grafos

Solução em nível intermediário. Código em inglês e explicações em PT-BR. A ideia foi seguir os exemplos das aulas e dos códigos de referência, sem deixar a solução exageradamente sofisticada.

## Implementação base

As classes abaixo são usadas ao longo dos exercícios.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        if word == "":
            return False

        current = self.root
        created_new = False

        for char in word:
            if char not in current.children:
                current.children[char] = TrieNode()
                created_new = True
            current = current.children[char]

        if current.is_end:
            return False

        current.is_end = True
        return True

    def search(self, word):
        if word == "":
            return False

        current = self.root

        for char in word:
            if char not in current.children:
                return False
            current = current.children[char]

        return current.is_end

    def starts_with(self, prefix):
        current = self.root

        for char in prefix:
            if char not in current.children:
                return False
            current = current.children[char]

        return True

    def find_node(self, prefix):
        current = self.root

        for char in prefix:
            if char not in current.children:
                return None
            current = current.children[char]

        return current

    def collect_words(self, node, prefix):
        words = []

        if node is None:
            return words

        if node.is_end:
            words.append(prefix)

        for char in node.children:
            child = node.children[char]
            words += self.collect_words(child, prefix + char)

        return words

    def all_words(self):
        return sorted(self.collect_words(self.root, ""))

    def words_from_prefix(self, prefix):
        node = self.find_node(prefix)
        return sorted(self.collect_words(node, prefix))

    def autocomplete(self, prefix, k):
        if k <= 0:
            return []

        suggestions = self.words_from_prefix(prefix)
        return suggestions[:k]

    def autocorrect(self, word):
        if self.search(word):
            return word

        current = self.root
        prefix = ""

        for char in word:
            if char not in current.children:
                break
            current = current.children[char]
            prefix += char

        candidates = sorted(self.collect_words(current, prefix))

        if len(candidates) == 0:
            return None

        return candidates[0]



In [ ]:
class GraphAdjList:
    def __init__(self):
        self.adj = {}

    def add_vertex(self, vertex):
        if vertex not in self.adj:
            self.adj[vertex] = []

    def add_edge(self, u, v, directed=False):
        self.add_vertex(u)
        self.add_vertex(v)

        if v not in self.adj[u]:
            self.adj[u].append(v)

        if not directed:
            if u not in self.adj[v]:
                self.adj[v].append(u)

    def has_edge(self, u, v):
        if u not in self.adj:
            return False
        return v in self.adj[u]

    def print_adj_list(self):
        for vertex in sorted(self.adj):
            print(vertex, "->", sorted(self.adj[vertex]))

    def vertices(self):
        return list(self.adj.keys())

    def to_mermaid(self, directed=False):
        if directed:
            connector = "-->"
        else:
            connector = "---"

        lines = ["graph TD"]
        used_edges = set()

        for u in sorted(self.adj):
            if len(self.adj[u]) == 0:
                lines.append("    " + u)

            for v in sorted(self.adj[u]):
                if directed:
                    lines.append("    " + u + " " + connector + " " + v)
                else:
                    edge = tuple(sorted([u, v]))
                    if edge not in used_edges:
                        used_edges.add(edge)
                        lines.append("    " + u + " " + connector + " " + v)

        return "\n".join(lines)


class GraphAdjMatrix:
    def __init__(self):
        self.index = {}
        self.vertices_names = []
        self.mat = []

    def add_vertex(self, vertex):
        if vertex in self.index:
            return

        self.index[vertex] = len(self.vertices_names)
        self.vertices_names.append(vertex)

        for row in self.mat:
            row.append(0)

        new_row = [0] * len(self.vertices_names)
        self.mat.append(new_row)

    def add_edge(self, u, v, directed=False):
        self.add_vertex(u)
        self.add_vertex(v)

        i = self.index[u]
        j = self.index[v]
        self.mat[i][j] = 1

        if not directed:
            self.mat[j][i] = 1

    def has_edge(self, u, v):
        if u not in self.index or v not in self.index:
            return False

        i = self.index[u]
        j = self.index[v]
        return self.mat[i][j] == 1

    def print_matrix(self):
        print("vertices:", self.vertices_names)
        for row in self.mat:
            print(row)


In [ ]:
EDGE = [
    ("auth", "user"),
    ("auth", "payment"),
    ("user", "profile"),
    ("user", "email"),
    ("profile", "report"),
    ("order", "product"),
    ("order", "payment"),
    ("product", "stock"),
    ("product", "search"),
    ("stock", "report"),
    ("search", "report"),
    ("email", "report"),
]


def create_graph(choice):
    if choice == "list":
        return GraphAdjList()
    elif choice == "matrix":
        return GraphAdjMatrix()
    else:
        raise ValueError("Opção inválida. Escolha 'list' ou 'matrix'.")


def build_sample_graph(choice):
    graph = create_graph(choice)
    edges = EDGE

    for u, v in edges:
        graph.add_edge(u, v)

    return graph


def find_vertices_by_prefix(prefix, k, trie, graph):
    candidates = trie.autocomplete(prefix, k)
    graph_vertices = graph.vertices()

    result = []
    for name in candidates:
        if name in graph_vertices:
            result.append(name)

    return result

In [ ]:
def run_basic_tests():
    trie = Trie()

    print("Exercício 1")
    print("inserir 'car':", trie.insert("car"), "esperado: True")
    print("inserir 'car' novamente:", trie.insert("car"), "esperado: False")
    print("inserir 'cart':", trie.insert("cart"), "esperado: True")
    print("inserir 'carro':", trie.insert("carro"), "esperado: True")

    print("\nExercício 2")
    print("buscar 'car':", trie.search("car"), "esperado: True")
    print("buscar 'cart':", trie.search("cart"), "esperado: True")
    print("buscar 'ca':", trie.search("ca"), "esperado: False")
    print("buscar 'casa':", trie.search("casa"), "esperado: False")
    print("buscar palavra vazia:", trie.search(""), "esperado: False")

    words = ["car", "cart", "carro", "cat", "cab", "cable", "dog", "door", "dove"]
    trie = Trie()

    for word in words:
        trie.insert(word)

    print("\nExercício 3")
    prefixes = ["ca", "car", "do", "z", "cab"]

    for prefix in prefixes:
        print("prefixo", prefix, "->", trie.starts_with(prefix))

    print("\nExercício 4")
    print("todas as palavras:", trie.all_words())
    print("palavras a partir do prefixo 'ca':", trie.words_from_prefix("ca"))

    print("\nExercício 5")
    print("autocomplete para 'ca', limite 3:", trie.autocomplete("ca", 3))
    print("autocomplete para 'dog', limite 5:", trie.autocomplete("dog", 5))
    print("autocomplete para 'x', limite 5:", trie.autocomplete("x", 5))

    print("\nExercício 6")
    tests = ["car", "carx", "cot", "dzz", "xyz", "ca"]

    for word in tests:
        print("autocorreção de", word, "->", trie.autocorrect(word))

    print("\nExercício 7")
    graph_list = build_sample_graph("list")
    print("Lista de adjacência do grafo:")
    graph_list.print_adj_list()

    print("\nExercício 8")
    graph_matrix = build_sample_graph("matrix")
    print("Matriz de adjacência do grafo:")
    graph_matrix.print_matrix()

    print("\nExercício 9")
    print("lista possui aresta auth-user:", graph_list.has_edge("auth", "user"))
    print("matriz possui aresta auth-user:", graph_matrix.has_edge("auth", "user"))
    print("lista possui aresta auth-report:", graph_list.has_edge("auth", "report"))
    print("matriz possui aresta auth-report:", graph_matrix.has_edge("auth", "report"))

    print("\nExercício 10")
    print("Representação Mermaid do grafo:")
    print(graph_list.to_mermaid())

    print("\nExercício 11")
    vertex_trie = Trie()

    for vertex in graph_list.vertices():
        vertex_trie.insert(vertex)

    print("vértices com prefixo 'p':", find_vertices_by_prefix("p", 5, vertex_trie, graph_list))
    print("vértices com prefixo 'pro':", find_vertices_by_prefix("pro", 5, vertex_trie, graph_list))
    print("vértices com prefixo 'x':", find_vertices_by_prefix("x", 5, vertex_trie, graph_list))


if __name__ == "__main__":
    run_basic_tests()

## Exercício 1 — Inserção e fim de palavra

A Trie usa `children` para guardar os filhos de cada nó e `is_end` para marcar quando uma palavra termina. Isso permite guardar `car`, `cart` e `carro` ao mesmo tempo.

In [ ]:
trie = Trie()
print(f"inserir 'car': {trie.insert("car")},", "esperado: True")
print(f"inserir 'car' novamente: {trie.insert("car")},", "esperado: False")
print(f"inserir 'cart': {trie.insert("car")},", "esperado: True")
print(f"inserir 'carro': {trie.insert("car")},", "esperado: True")

inserir 'car': True, esperado: True
inserir 'car' novamente: False, esperado: False
inserir 'cart': False, esperado: True
inserir 'carro': False, esperado: True


## Exercício 2 — Busca exata

A busca exata só retorna `True` quando o caminho existe e o último nó está com `is_end = True`. Eu decidi não considerar palavra vazia como palavra válida.

In [ ]:
print("\nExercício 2")
print(f"buscar 'car': {trie.search("car")},", "esperado: True")
print(f"buscar 'cart': {trie.search("cart")},", "esperado: True")
print(f"buscar 'ca': {trie.search("ca")},", "esperado: False")
print(f"buscar 'casa': {trie.search("casa")},", "esperado: False")
print(f"buscar palavra vazia: {trie.search("")},", "esperado: False")


Exercício 2
buscar 'car': True, esperado: True
buscar 'cart': False, esperado: True
buscar 'ca': False, esperado: False
buscar 'casa': False, esperado: False
buscar palavra vazia: False, esperado: False


## Exercício 3 — Prefixos

Aqui basta existir o caminho. Não precisa ser palavra completa.

In [ ]:
words = ["car", "cart", "carro", "cat", "cab", "cable", "dog", "door", "dove"]
trie = Trie()
for word in words:
    trie.insert(word)

for prefix in ["a", "ba", "ca", "car", "do", "z", "cab"]:
    print(prefix, "->", trie.starts_with(prefix))

a -> False
ba -> False
ca -> True
car -> True
do -> True
z -> False
cab -> True


## Exercício 4 — Traversal

O método `collect_words` percorre uma subárvore e monta as palavras encontradas. Usei `sorted` na saída para facilitar a comparação.

In [ ]:
print("all words:", trie.all_words())
print("words from ca:", trie.words_from_prefix("ca"))
print("same words?", trie.all_words() == sorted(set(words)))

all words: ['cab', 'cable', 'car', 'carro', 'cart', 'cat', 'dog', 'door', 'dove']
words from ca: ['cab', 'cable', 'car', 'carro', 'cart', 'cat']
same words? True


## Exercício 5 — Autocomplete

O autocomplete procura o nó do prefixo, coleta as palavras abaixo dele, ordena e devolve no máximo `k` sugestões.

In [ ]:
print(trie.autocomplete("ca", 3))
print(trie.autocomplete("dog", 5))
print(trie.autocomplete("x", 5))

['cab', 'cable', 'car']
['dog']
[]


## Exercício 6 — Autocorrect

O autocorrect aqui é simples: usa o maior prefixo que ainda existe na Trie e, em empate, escolhe a menor palavra lexicográfica.

In [ ]:
for word in ["car", "carx", "cot", "dzz", "xyz", "ca"]:
    print(word, "->", trie.autocorrect(word))

car -> car
carx -> car
cot -> car
dzz -> car
xyz -> car
ca -> car


## Exercício 7 — Grafo por lista

A lista de adjacência guarda, para cada vértice, uma lista com seus vizinhos. O grafo tem 10 vértices e 12 arestas.

In [ ]:
graph_list = build_sample_graph("list")
graph_list.print_adj_list()

auth -> ['payment', 'user']
email -> ['report', 'user']
order -> ['payment', 'product']
payment -> ['auth', 'order']
product -> ['order', 'search', 'stock']
profile -> ['report', 'user']
report -> ['email', 'profile', 'search', 'stock']
search -> ['product', 'report']
stock -> ['product', 'report']
user -> ['auth', 'email', 'profile']


## Exercício 8 — Grafo por matriz

A matriz de adjacência usa `0` e `1`. Também guardei um dicionário `index` para saber qual linha/coluna pertence a cada vértice.

In [ ]:
graph_matrix = build_sample_graph("matrix")
graph_matrix.print_matrix()

vertices: ['auth', 'user', 'payment', 'profile', 'email', 'report', 'order', 'product', 'stock', 'search']
[0, 1, 1, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 1, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 1, 0, 0, 0]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 0, 0, 1, 1, 0, 0, 0, 1, 1]
[0, 0, 1, 0, 0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0, 1, 1]
[0, 0, 0, 0, 0, 1, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 1, 0, 1, 0, 0]


## Exercício 9 — Definições e consultas

Vértice é cada entidade do grafo. Aresta é a ligação entre duas entidades. Na lista, a aresta aparece na lista de vizinhos. Na matriz, aparece como valor 1 na posição correspondente.

In [ ]:
print("list auth-user:", graph_list.has_edge("auth", "user"))
print("matrix auth-user:", graph_matrix.has_edge("auth", "user"))
print("list auth-report:", graph_list.has_edge("auth", "report"))
print("matrix auth-report:", graph_matrix.has_edge("auth", "report"))

list auth-user: True
matrix auth-user: True
list auth-report: False
matrix auth-report: False


## Exercício 10 — Mermaid

Para evitar duplicação em grafo não direcionado, uso uma tupla ordenada da aresta dentro de `used_edges`.

In [ ]:
print(graph_list.to_mermaid())

graph TD
    auth --- payment
    auth --- user
    email --- report
    email --- user
    order --- payment
    order --- product
    product --- search
    product --- stock
    profile --- report
    profile --- user
    report --- search
    report --- stock


## Exercício 11 — Trie como índice dos vértices

Coloquei os nomes dos vértices na Trie e depois usei autocomplete para buscar vértices por prefixo.

In [ ]:
vertex_trie = Trie()
for vertex in graph_list.vertices():
    vertex_trie.insert(vertex)

print(find_vertices_by_prefix("p", 5, vertex_trie, graph_list))
print(find_vertices_by_prefix("pro", 5, vertex_trie, graph_list))
print(find_vertices_by_prefix("x", 5, vertex_trie, graph_list))

['payment', 'product', 'profile']
['product', 'profile']
[]


## Exercício 12 — Decisões de Projeto e Complexidade

### Operações da Trie

| Operação | Custo aproximado | Comentário |
|---|---:|---|
| `insert(word)` | `O(k)` | percorre os caracteres da palavra |
| `search(word)` | `O(k)` | percorre e confere `is_end` |
| `starts_with(prefix)` | `O(p)` | só verifica se caminho existe |
| `collect_words(node, prefix)` | `O(m)` | percorre a subárvore |
| `autocomplete(prefix, k)` | `O(p + m + s log s)` | busca prefixo, coleta e ordena |
| `autocorrect(word)` | `O(k + m + s log s)` | acha prefixo comum e escolhe sugestão |

### Lista x matriz

| Aspecto | Lista de adjacência | Matriz de adjacência |
|---|---:|---:|
| Inserção de aresta | `O(grau)` na minha versão | `O(1)` depois dos vértices |
| Consulta de aresta | `O(grau)` | `O(1)` |
| Memória | `O(V + E)` | `O(V²)` |

Para grafo esparso com milhões de vértices, eu escolheria lista de adjacência por causa da memória. Para grafo denso com poucos vértices, escolheria matriz, porque a consulta é direta e o custo de memória ainda é aceitável.

Uma decisão da Trie foi usar `children` como dicionário e `is_end` para separar prefixo de palavra completa. Uma decisão do grafo foi usar lista de vizinhos, porque fica parecido com os exemplos da aula, mesmo que `set` fosse melhor para consulta.
